In [0]:
spark

### Comunicação Hive-Spark

In [0]:
%sql
show tables

In [0]:
tabela = spark.table('tabela_vinhos.combined_wines')

In [0]:
tabela --ver o dataframe criado

In [0]:
tabela.show(); --visualizo o conteúdo

In [0]:
display(tabela) -- visualização melhor.

### SQL com Spark

`spark.sql('query').show()`
ou
`display(spark.sql('query'))`

Se quisermos pular linhas na query temos que utilizas 3 aspas simples

`spark.sql('''
    query
''').show()`

In [0]:
spark.sql('''
          select distinct (quality)
          from tabela_vinhos.combined_wines
          order by quality desc
          ''').show()

In [0]:
spark.sql('select avg(pH) from combined_wines').show()

### Registrando uma tabela

In [0]:
##criando uma temp view, primeiro passo criar o dataframe
resultado = spark.sql('''
    select * from combined_wines
    where ph < 3
''')

In [0]:
type(resultado)

In [0]:
##criando a temp view
resultado.createOrReplaceTempView('nova_tabela')

In [0]:
spark.sql('''
          select quality, count(quality) as freq
          from nova_tabela
          group by quality
          ''').show()

### PySpark

In [0]:
import pyspark
from pyspark.sql.functions import lit

In [0]:
display(dbutils.fs.ls('/databricks-datasets/wine-quality'))

In [0]:
##criando um dataframe
red_wine_df = spark.read.format('csv')\
    .option('inferSchema', 'true')\
    .option('delimiter', ';')\
    .option('header', 'true')\
    .load('/databricks-datasets/wine-quality/winequality-red.csv')

display(red_wine_df)

In [0]:
type(red_wine_df)

In [0]:
##criando um dataframe
white_wine_df = spark.read.format('csv')\
    .option('inferSchema', 'true')\
    .option('delimiter', ';')\
    .option('header', 'true')\
    .load('/databricks-datasets/wine-quality/winequality-white.csv')

display(white_wine_df)

In [0]:
red_wine_df = red_wine_df.withColumn('wine_type', lit('red'))
red_wine_df.show()

In [0]:
white_wine_df = white_wine_df.withColumn('wine_type', lit('white'))
white_wine_df.show()

In [0]:
combine_wines = red_wine_df.union(white_wine_df)
display(combine_wines)

In [0]:
##mudar o nome da coluna
combine_wines = combine_wines.withColumnRenamed('quality', 'nota')
display(combine_wines)

In [0]:
(
    combine_wines
    .select(['nota', 'wine_type'])
    .show()
)

In [0]:
(
    combine_wines
        .groupBy(['nota', 'wine_type'])
        .count()
        .show()
)

In [0]:
combine_wines.printSchema()

In [0]:
##salvar o dataframe em um csv dentro do databricks mesmo
(
    combine_wines
    .write
    .option('header', 'true')
    .mode('overwrite')
    .csv('/FileStore/tables/aula-databricks/vinhos/pyspark/wine_quality')
)